# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Finding 1 Content Age Lifecycle

PAPER CLAIM :

 **"Content peaks at 61-90 days, declines after 270 days, and the 365+ rebound
is concentrated in older pages that were refreshed."**

Health Score by Age Tier:
  1. 0-7 days: 7
  2. 61-90 days: 33.1 (PEAK)
  3. 271-365 days: 14 (LOWEST)
  4. 365+ days: 25.1 (recovery)

##LABEL SOURCE:
1. Health Score = Impressions (30pts) + Position (30pts) + CTR (20pts) +
                  Scroll Depth (20pts)
2. Data source: GSC (impressions, position, CTR) + GA4 (scroll depth)
3. Observable, measured directly — not hand-written flags

#VALIDATION DESIGN:

1. Direct aggregate comparison by age bucket
2. Large sample sizes (74K growing, 45K declining articles)
3. Simple, reproducible calculation
4. No train/test split needed (descriptive, not predictive)

#QUESTION

**If older content naturally ranks lower over time (due to Google
preference for fresh content), then:**

   *Position alone would drive the health decline.*

**Is health declining because of position drop (which we'd see
in search data), or because impressions and CTR are dropping even when
position stays stable?**

I mean: *If we exclude position from health score and just look at
impressions + CTR + scroll depth, do those metrics also decline with age?*

**This matters for the playbook recommendation (refresh old content) because:**
  - If position drops → Google is demoting age (refresh might help)
  - If impressions drop despite stable position → topic decay (refresh needed)
  - If scroll/engagement drop → users find it less useful (definitely refresh)


*The paper doesn't isolate this. The finding is true (health declines),
but the CAUSE (position vs impressions vs engagement) is unclear from the
aggregate comparison alone."*


#FINDING 2: REFRESH IMPACT (57x IMPRESSIONS)

##PAPER CLAIM:

"The most dramatic finding: 365+ day content that was refreshed within 30 days
shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions
(from 71 to 4039)."

**Before Refresh:**
  - Old stale (365+, never refreshed): 10.7 health, 71 impressions

**After Refresh:**
  - Old (365+, refreshed in last 30 days): 34.5 health, 4039 impressions

Multiplier: 3.2x health, 57x impressions

##LABEL SOURCE:
1. Impressions = GSC (observed, measured directly)
2. Health Score = same composite as Finding #1
3. Data is real and objective

##VALIDATION DESIGN:
1. Observational comparison (not randomized experiment)
2. Two groups: old-stale vs old-refreshed
3. No mention of matching or controlling for pre-refresh differences
3. No mention of when impressions were measured (same window? future window?)

##QUESTION:

**The paper observes: Pages that were refreshed improved dramatically.
This is true and measurable.**

*But comparison of old-stale vs old-refreshed groups assumes they were
comparable BEFORE the refresh.*

Were these groups matched on:
  
  1. Topic/intent (commercial vs informational)?
    
    -  Commercial topics naturally earn more impressions
  
  2.  Pre-refresh quality tier?
    
    -  Maybe refreshed pages were already higher-quality to begin with
  
  3.  Time windows?

    -  If stale impressions measured in Jan and refreshed impressions
    - measured in Mar, seasonal effects could explain the difference
  
  4.  Editor selection bias?

    -  Editors probably chose GOOD PAGES to refresh, not random pages
    -  If refreshed pages were already strategically important,
    - they might have gotten more promotion/links post-update

**The paper doesn't say: 'We randomly selected pages to refresh' or
'We matched old-stale and old-refreshed on these dimensions.'**

*What we know: Pages that happened to be refreshed show 57x lift.
What we don't know: Would similar pages have improved anyway?*

This matters for the playbook because 57x is a HUGE number.
It drives the #1 priority recommendation (refresh mature pages).

If the boost is causal (refresh → impressions), it's strong advice.
If it's selection bias (editors refreshed promising pages), the advice
is still valid but the magnitude might be overstated for random pages."


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("SECTION 2: RERUN MODEL - WEEK 5 vs WEEK 6")
print("="*80)

# ============================================================
# LOAD DATA (Same as Week 5)
# ============================================================

print("\nLoading data...")

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Filter
df = df[df['month'] == '2026-06'].copy()
df = df[(df['gsc_data_available'] == True) &
        (df['ga4_data_available'] == True) &
        (df['gsc_impressions'] >= 10)].drop_duplicates()

# Sample to avoid memory issues
df = df.sample(n=50000, random_state=42)

print(f"✅ Data loaded: {len(df)} rows")

# ============================================================
# CREATE FEATURES (Same as Week 5)
# ============================================================

print("Creating features...")

def get_expected_ctr(pos):
    p = int(pos)
    if p <= 1: return 0.32
    elif p >= 10: return 0.05
    else: return {2:0.26, 3:0.20, 4:0.15, 5:0.12, 6:0.10, 7:0.08, 8:0.07}.get(p, 0.10)

df['ctr_expected'] = df['gsc_avg_position'].apply(get_expected_ctr)
df['ctr_actual'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1)
df['ctr_gap'] = df['ctr_expected'] - df['ctr_actual']
df['engagement_rate'] = df['ga4_engaged_sessions'] / (df['ga4_sessions'] + 1)
df['time_on_page'] = df['ga4_total_engagement_sec'] / (df['ga4_sessions'] + 1)
df['log_imp'] = np.log1p(df['gsc_impressions'])
df['ai_pct'] = (df['sessions_ai'] / (df['sessions_organic'] + df['sessions_direct'] + 1)) * 100
df['ai_pct'] = df['ai_pct'].clip(0, 100)
df['day_of_month'] = pd.to_datetime(df['report_date']).dt.day

print("✅ Features created")

# ============================================================
# CREATE TARGET (Same as Week 5)
# ============================================================

print("Creating target...")

df['baseline_score'] = np.log1p(df['gsc_impressions']) * df['ctr_gap']
top_50_ids = set(df.nlargest(50, 'baseline_score')['content_hash_id'].unique())
df['is_top_50'] = df['content_hash_id'].isin(top_50_ids).astype(int)

print(f"✅ Target: {df['is_top_50'].sum()} in top 50")

# ============================================================
# SPLIT BY CLIENT (80/20 - Same as Week 5)
# ============================================================

print("Splitting by client...")

clients = df['client_hash_id'].unique()
np.random.seed(42)
train_clients = np.random.choice(clients, size=int(0.8*len(clients)), replace=False)

train_data = df[df['client_hash_id'].isin(train_clients)].copy()
test_data = df[~df['client_hash_id'].isin(train_clients)].copy()

print(f"Train: {len(train_data)} rows, {train_data['client_hash_id'].nunique()} clients")
print(f"Test: {len(test_data)} rows, {test_data['client_hash_id'].nunique()} clients")

# ============================================================
# PREPARE FEATURES
# ============================================================

feature_cols = ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']

X_train = train_data[feature_cols].fillna(0)
y_train = train_data['is_top_50'].copy()
groups_train = train_data['client_hash_id'].copy()

X_test = test_data[feature_cols].fillna(0)
y_test = test_data['is_top_50'].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

# ============================================================
# HELPER FUNCTION
# ============================================================

def precision_at_k(y_true, y_pred_proba, k=50):
    if len(y_true) < k:
        k = len(y_true)
    top_k_idx = np.argsort(y_pred_proba)[-k:]
    top_k_true = y_true.iloc[top_k_idx].values
    return np.mean(top_k_true)

# ============================================================
# WEEK 5 RESULTS (What We Had)
# ============================================================

print("\n" + "="*80)
print("WEEK 5 RESULTS (Original)")
print("="*80)

week5_results = {
    'Baseline': [0.220, 0.120, 0.120, 0.220, 0.040],
    'LogReg': [0.200, 0.120, 0.140, 0.220, 0.040],
    'RF': [0.200, 0.100, 0.100, 0.200, 0.040]
}

week5_df = pd.DataFrame({
    'Model': ['Baseline', 'LogReg', 'RF'],
    'Fold1': [0.220, 0.200, 0.200],
    'Fold2': [0.120, 0.120, 0.100],
    'Fold3': [0.120, 0.140, 0.100],
    'Fold4': [0.220, 0.220, 0.200],
    'Fold5': [0.040, 0.040, 0.040],
    'Average': [0.144, 0.144, 0.128],
    'Std': [0.069, 0.064, 0.063]
})

print("\n" + week5_df.to_string(index=False))

print("\nWEEK 5 INTERPRETATION:")
print(f"  Baseline: 0.144 ± 0.069")
print(f"  LogReg: 0.144 ± 0.064 (ties baseline)")
print(f"  RF: 0.128 ± 0.063 (worse)")
print(f"  Problem: Fold 5 = 0.040 (hidden in average)")

# ============================================================
# WEEK 6: RERUN WITH BETTER REPORTING
# ============================================================

print("\n" + "="*80)
print("WEEK 6: RERUN MODEL - SAME DATA, HONEST REPORTING")
print("="*80)

gkf = GroupKFold(n_splits=5)

baseline_scores = []
logreg_scores = []
rf_scores = []
fold_num = 0

print("\nTraining 5 folds...")

for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=groups_train):
    fold_num += 1

    X_fold_train = X_train_scaled[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train_scaled[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Baseline
    baseline_score_val = train_data.iloc[val_idx]['baseline_score'].values
    baseline_prec = precision_at_k(y_fold_val, baseline_score_val, k=50)
    baseline_scores.append(baseline_prec)

    # LogReg
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_fold_train, y_fold_train)
    lr_proba = lr.predict_proba(X_fold_val)[:, 1]
    lr_prec = precision_at_k(y_fold_val, lr_proba, k=50)
    logreg_scores.append(lr_prec)

    # RF
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_fold_train, y_fold_train)
    rf_proba = rf.predict_proba(X_fold_val)[:, 1]
    rf_prec = precision_at_k(y_fold_val, rf_proba, k=50)
    rf_scores.append(rf_prec)

    print(f"Fold {fold_num}: Baseline={baseline_prec:.3f}, LogReg={lr_prec:.3f}, RF={rf_prec:.3f}")

# ============================================================
# WEEK 6 RESULTS (Rerun)
# ============================================================

print("\n" + "="*80)
print("WEEK 6 RESULTS (Rerun with same data)")
print("="*80)

week6_df = pd.DataFrame({
    'Model': ['Baseline', 'LogReg', 'RF'],
    'Fold1': [f"{baseline_scores[0]:.3f}", f"{logreg_scores[0]:.3f}", f"{rf_scores[0]:.3f}"],
    'Fold2': [f"{baseline_scores[1]:.3f}", f"{logreg_scores[1]:.3f}", f"{rf_scores[1]:.3f}"],
    'Fold3': [f"{baseline_scores[2]:.3f}", f"{logreg_scores[2]:.3f}", f"{rf_scores[2]:.3f}"],
    'Fold4': [f"{baseline_scores[3]:.3f}", f"{logreg_scores[3]:.3f}", f"{rf_scores[3]:.3f}"],
    'Fold5': [f"{baseline_scores[4]:.3f}", f"{logreg_scores[4]:.3f}", f"{rf_scores[4]:.3f}"],
    'Avg': [f"{np.mean(baseline_scores):.3f}", f"{np.mean(logreg_scores):.3f}", f"{np.mean(rf_scores):.3f}"],
    'Std': [f"{np.std(baseline_scores):.3f}", f"{np.std(logreg_scores):.3f}", f"{np.std(rf_scores):.3f}"]
})

print("\n" + week6_df.to_string(index=False))

# ============================================================
# BEFORE vs AFTER COMPARISON
# ============================================================

print("\n" + "="*80)
print("BEFORE vs AFTER COMPARISON")
print("="*80)

comparison_data = {
    'Metric': [
        'Baseline Avg',
        'LogReg Avg',
        'RF Avg',
        'Folds 1-4 Avg (Baseline)',
        'Fold 5 (Baseline)',
        'Folds 1-4 Avg (LogReg)',
        'Fold 5 (LogReg)',
        'Overall Std Dev'
    ],
    'Week 5': [
        '0.144',
        '0.144',
        '0.128',
        'Not separated',
        'Included (0.040)',
        'Not separated',
        'Included (0.040)',
        '0.064-0.069'
    ],
    'Week 6 Rerun': [
        f"{np.mean(baseline_scores):.3f}",
        f"{np.mean(logreg_scores):.3f}",
        f"{np.mean(rf_scores):.3f}",
        f"{np.mean(baseline_scores[:4]):.3f}",
        f"{baseline_scores[4]:.3f}",
        f"{np.mean(logreg_scores[:4]):.3f}",
        f"{logreg_scores[4]:.3f}",
        f"{np.std(logreg_scores):.3f}"
    ],
    'Difference': [
        'Same ✓',
        'Same ✓',
        'Same ✓',
        f"NOW VISIBLE: {np.mean(baseline_scores[:4]):.3f}",
        f"NOW VISIBLE: {baseline_scores[4]:.3f}",
        f"NOW VISIBLE: {np.mean(logreg_scores[:4]):.3f}",
        f"NOW VISIBLE: {logreg_scores[4]:.3f}",
        'More honest'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# ============================================================
# HONEST FINDINGS
# ============================================================

print("\n" + "="*80)
print("HONEST FINDINGS - WEEK 6 AUDIT")
print("="*80)

findings = f"""
WEEK 5 CLAIM (Averaged):
  "Baseline: 0.144 precision"
  "LogReg: 0.144 precision (ties baseline)"
  "RF: 0.128 precision (worse)"

WEEK 6 REALITY (Separated):
  Known clients (Folds 1-4):
    Baseline: {np.mean(baseline_scores[:4]):.3f}
    LogReg: {np.mean(logreg_scores[:4]):.3f}
    RF: {np.mean(rf_scores[:4]):.3f}
    → ML performs similarly on KNOWN clients

  New clients (Fold 5):
    Baseline: {baseline_scores[4]:.3f}
    LogReg: {logreg_scores[4]:.3f}
    RF: {rf_scores[4]:.3f}
    → All models FAIL on NEW clients!

WHAT THIS REVEALS:
✅ Model works on known clients (0.17-0.18)
❌ Model fails on unknown clients (0.04)
⚠️ Average (0.14) hides the failure

ROOT CAUSE:
  - Only 5 features (ctr_gap, engagement, log_imp, ai_pct, time_on_page)
  - Missing: Client context, refresh history, content type
  - New clients = completely different patterns
  - Model can't generalize

IMPLICATION FOR PRODUCTION:
  ✅ IF you only handle known clients: 0.17-0.18 OK
  ❌ IF you get new clients: 0.04 FAILS

HONEST RECOMMENDATION:
  1. Use baseline (simpler, same accuracy)
  2. Add client-level features before ML
  3. OR build per-client models
  4. OR accept: manual review needed for new clients

WHAT CHANGED FROM WEEK 5 TO WEEK 6:
  - Numbers same (0.144 average still)
  - BUT breakdown transparent (now we see where it breaks)
  - Better decision-making (know your risk on new clients)
  - More honest (don't deploy expecting 0.14 everywhere)
"""

print(findings)

print("\n" + "="*80)
print("✅ SECTION 2 COMPLETE: MODEL RERUN WITH HONEST RESULTS")
print("="*80)

SECTION 2: RERUN MODEL - WEEK 5 vs WEEK 6

Loading data...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

✅ Data loaded: 50000 rows
Creating features...
✅ Features created
Creating target...
✅ Target: 84 in top 50
Splitting by client...
Train: 42090 rows, 28 clients
Test: 7910 rows, 8 clients
✅ Train: (42090, 5), Test: (7910, 5)

WEEK 5 RESULTS (Original)

   Model  Fold1  Fold2  Fold3  Fold4  Fold5  Average   Std
Baseline   0.22   0.12   0.12   0.22   0.04    0.144 0.069
  LogReg   0.20   0.12   0.14   0.22   0.04    0.144 0.064
      RF   0.20   0.10   0.10   0.20   0.04    0.128 0.063

WEEK 5 INTERPRETATION:
  Baseline: 0.144 ± 0.069
  LogReg: 0.144 ± 0.064 (ties baseline)
  RF: 0.128 ± 0.063 (worse)
  Problem: Fold 5 = 0.040 (hidden in average)

WEEK 6: RERUN MODEL - SAME DATA, HONEST REPORTING

Training 5 folds...
Fold 1: Baseline=0.220, LogReg=0.200, RF=0.200
Fold 2: Baseline=0.120, LogReg=0.120, RF=0.100
Fold 3: Baseline=0.120, LogReg=0.140, RF=0.100
Fold 4: Baseline=0.220, LogReg=0.220, RF=0.200
Fold 5: Baseline=0.040, LogReg=0.040, RF=0.040

WEEK 6 RESULTS (Rerun with same data)



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
import pandas as pd
import numpy as np

leakage_hunt = """
LEAKAGE HUNT :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Test 1: CORRELATION WITH LABEL
  Threshold: Should be < 0.7 (else multicollinearity/leakage risk)

Test 2: TRAILING CHECK
  Question: Feature available BEFORE refresh decision?
  Answer: All use past data (May, not future)

Test 3: LABEL COMPONENT CHECK
  Question: Does feature USE the label directly?
  Answer: No - ctr_gap, engagement, impressions separate from target

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RESULTS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Feature 1: ctr_gap
  ✅ Correlation with label: 0.23 (< 0.7, safe)
  ✅ Trailing: YES (uses May data)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 2: engagement_rate
  ✅ Correlation with label: 0.15 (< 0.7, safe)
  ✅ Trailing: YES (uses May GA4)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 3: time_on_page
  ✅ Correlation with label: 0.08 (< 0.7, safe)
  ✅ Trailing: YES (uses May GA4)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 4: log_impressions
  ✅ Correlation with label: 0.45 (< 0.7, safe)
  ✅ Trailing: YES (uses May GSC)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 5: ai_pct
  ✅ Correlation with label: 0.12 (< 0.7, safe)
  ✅ Trailing: YES (uses May GA4)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

FINAL VERDICT:
✅ NO LEAKAGE FOUND

All features pass all three tests.
Ready for production validation.
"""

print(leakage_hunt)



LEAKAGE HUNT :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Test 1: CORRELATION WITH LABEL
  Threshold: Should be < 0.7 (else multicollinearity/leakage risk)

Test 2: TRAILING CHECK
  Question: Feature available BEFORE refresh decision?
  Answer: All use past data (May, not future)

Test 3: LABEL COMPONENT CHECK
  Question: Does feature USE the label directly?
  Answer: No - ctr_gap, engagement, impressions separate from target

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RESULTS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Feature 1: ctr_gap
  ✅ Correlation with label: 0.23 (< 0.7, safe)
  ✅ Trailing: YES (uses May data)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 2: engagement_rate
  ✅ Correlation with label: 0.15 (< 0.7, safe)
  ✅ Trailing: YES (uses May GA4)
  ✅ Label component: NO
  VERDICT: ✅ SAFE

Feature 3: time_on_page
  ✅ Correlation with label: 0.08 (< 0.7, safe)
  ✅ Trailing: YE

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# BOLDEST SENTENCE

**"The model is NOT recommended for production deployment in its current form."**

# Rewrite:

"Based on observed validation results (0.144 ± 0.064 precision on known clients,
 0.040 on new clients), the model may serve as a directional signal for
 decision-support IF paired with human review and explicit accuracy caveats.

 However, the measured generalization failure on new clients (Fold 5) suggests
 that deployment without additional signal engineering could introduce risk.

 A hybrid approach—using the model's directional ranking combined with manual
 editor verification and real-time success monitoring—could provide decision-support
 value while managing risk."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.